# Agentic RAG explainer and demo

## Setup instructions

1. [Install uv](https://docs.astral.sh/uv/getting-started/installation/) if not already installed
2. Open a terminal and run `uv sync`
3. Create a file called `.env` and add one line: `OPENAI_API_KEY=sk-proj-xxx`

## PART 1: Traditional RAG refresher


In [ ]:
from dotenv import load_dotenv
import json
from chromadb import PersistentClient
from openai import OpenAI
from sklearn.manifold import TSNE
import numpy as np
import plotly.graph_objects as go

MODEL_NAME = 'gpt-5.4-mini'
DB_NAME = 'vectorstore'
collection_name = 'facts'
embedding_model = "text-embedding-3-small"

load_dotenv(override=True)

In [ ]:
openai = OpenAI()

In [ ]:
with open('facts.jsonl', 'r', encoding='utf-8') as f:
    facts = [json.loads(line) for line in f]

print(f"Loaded {len(facts)} facts starting with: {facts[0]}")

## Put all the facts in a Chroma vectorstore

Common misconception: even though we call it a "vectorstore", the key information being stored is the actual text. The point is that the text is indexed off the vector, and can be easily retrieved based on vector similarity.

In [ ]:
chroma = PersistentClient(path=DB_NAME)

if collection_name in [c.name for c in chroma.list_collections()]:
    chroma.delete_collection(collection_name)

texts = [fact['fact'] for fact in facts]
results = openai.embeddings.create(model=embedding_model, input=texts).data
vectors = [r.embedding for r in results]


collection = chroma.get_or_create_collection(collection_name)

ids = [str(i) for i in range(len(facts))]
metas = [{'category': fact['category']} for fact in facts]

collection.add(ids=ids, embeddings=vectors, documents=texts, metadatas=metas)
print(f"Vectorstore created with {collection.count()} documents")

## I love to visualize data keyed off a vector

In [ ]:
categories = set(fact['category'] for fact in facts)
colors = ['red', 'blue', 'green', 'magenta']
color_map = {category: colors[i] for i, category in enumerate(categories)}

In [ ]:
result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
documents = result['documents']
categories = [metadata['category'] for metadata in result['metadatas']]
colors = [color_map[category] for category in categories]

In [ ]:
tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 2D scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Category: {c}<br>Text: {d[:100]}..." for c, d in zip(categories, documents)],
    hoverinfo='text'
)])

fig.update_layout(title='2D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x',yaxis_title='y'),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

In [ ]:
tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(categories, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='3D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900,
    height=700,
    margin=dict(r=10, b=10, l=10, t=40)
)

fig.show()

## And now it's time to ask a question

In [ ]:
question = "What is Ed's favorite fruit?"

In [ ]:
response = openai.chat.completions.create(model=MODEL_NAME, messages=[{"role": "user", "content": question}])
reply = response.choices[0].message.content
print(reply)

## Let's give our LLM some relevant context

In [ ]:
query = openai.embeddings.create(model=embedding_model, input=[question]).data[0].embedding
results = collection.query(query_embeddings=[query], n_results=3)
results["documents"]

In [ ]:
context = "\n".join(results["documents"][0])
question_with_context = f"""
This context might be useful:
{context}

Now with this context, please answer the following question:
{question}
"""

In [ ]:
print(question_with_context)

## And now - here is good old-fashioned RAG

In [ ]:
response = openai.chat.completions.create(model=MODEL_NAME, messages=[{"role": "user", "content": question_with_context}])
reply = response.choices[0].message.content
print(reply)

# Ta-da!